In [ ]:
import nltk
import numpy as np
from collections import Counter, defaultdict
import random
import math
nltk.download('treebank', quiet=True)
nltk.download('universal_tagset', quiet=True)

from nltk.corpus import treebank

In [ ]:
tagged_sents = treebank.tagged_sents(tagset='universal')

print(f"Total sentences in corpus: {len(tagged_sents)}")
print(f"\nSample sentence (first 8 tokens):")
for word, tag in tagged_sents[0][:8]:
    print(f"  {word}: {tag}")

# Collect all unique tags
all_tags = set()
total_tokens = 0
for sent in tagged_sents:
    for word, tag in sent:
        all_tags.add(tag)
        total_tokens += 1

print(f"\nTotal tokens: {total_tokens:,}")
print(f"Unique POS tags ({len(all_tags)}): {sorted(all_tags)}")

Total sentences in corpus: 3914

Sample sentence (first 8 tokens):
  Pierre: NOUN
  Vinken: NOUN
  ,: .
  61: NUM
  years: NOUN
  old: ADJ
  ,: .
  will: VERB

Total tokens: 100,676
Unique POS tags (12): ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']


In [ ]:
random.seed(42)
all_sents = list(tagged_sents)
random.shuffle(all_sents)

split_idx = int(0.8 * len(all_sents))
train_sents = all_sents[:split_idx]
test_sents = all_sents[split_idx:]

print(f"Training sentences: {len(train_sents)}")
print(f"Testing sentences:  {len(test_sents)}")

Training sentences: 3131
Testing sentences:  783


### HMM Tagger

In [ ]:
class HMMTagger:

    def __init__(self):
        self.tags = []
        self.tag_to_idx = {}
        self.vocab = set()

        self.transition_counts = None
        self.emission_counts = None
        self.initial_counts = None

        self.log_transition = None
        self.log_emission = None
        self.log_initial = None

        self.emission_denominators = None
        self.emission_tag_totals = None

    def _guess_tag_from_suffix(self, word, position_in_sentence):
        lower = word.lower()

        # 1. Words ending in -ing -> VERB
        if lower.endswith('ing'):
            return 'VERB'

        # 2. Words ending in -ly -> ADV
        if lower.endswith('ly'):
            return 'ADV'

        # 3. Words ending in -ness, -tion -> NOUN
        if lower.endswith('ness') or lower.endswith('tion'):
            return 'NOUN'

        # 4. Capitalised words -> PROPN
        if word[0].isupper():
            return 'PROPN'

        # Default fallback
        return 'NOUN'

    def train(self, tagged_sentences):

        tag_set = set()
        for sent in tagged_sentences:
            for word, tag in sent:
                tag_set.add(tag)
                self.vocab.add(word)

        self.tags = sorted(tag_set)
        self.tag_to_idx = {t: i for i, t in enumerate(self.tags)}
        num_tags = len(self.tags)
        vocab_size = len(self.vocab)

        print(f"Training HMM on {len(tagged_sentences)} sentences...")
        print(f"  Vocabulary size: {vocab_size:,}")
        print(f"  Number of tags:  {num_tags}")
        print(f"  Tags: {self.tags}")

        self.transition_counts = np.zeros((num_tags, num_tags), dtype=np.float64)

        self.emission_counts = defaultdict(lambda: defaultdict(float))

        self.initial_counts = np.zeros(num_tags, dtype=np.float64)

        tag_unigram_counts = np.zeros(num_tags, dtype=np.float64)

        for sent in tagged_sentences:
            for i, (word, tag) in enumerate(sent):
                tag_idx = self.tag_to_idx[tag]

                self.emission_counts[tag_idx][word] += 1
                tag_unigram_counts[tag_idx] += 1

                if i == 0:
                    self.initial_counts[tag_idx] += 1
                else:
                    prev_tag_idx = self.tag_to_idx[sent[i - 1][1]]
                    self.transition_counts[prev_tag_idx][tag_idx] += 1


        transition_row_sums = self.transition_counts.sum(axis=1)
        smoothed_transition = (self.transition_counts + 1) / (transition_row_sums[:, np.newaxis] + num_tags)
        self.log_transition = np.log(smoothed_transition)

        total_sentences = len(tagged_sentences)
        smoothed_initial = (self.initial_counts + 1) / (total_sentences + num_tags)
        self.log_initial = np.log(smoothed_initial)


        self.emission_tag_totals = tag_unigram_counts
        self.emission_denominators = tag_unigram_counts + vocab_size + 1  # +1 for OOV


        self.log_emission = {}
        for tag_idx in range(num_tags):
            self.log_emission[tag_idx] = {}
            denom = self.emission_denominators[tag_idx]
            for word, count in self.emission_counts[tag_idx].items():
                self.log_emission[tag_idx][word] = math.log((count + 1) / denom)

        # OOV log probability for each tag: log(1 / denominator)
        self.log_emission_oov = np.log(1.0 / self.emission_denominators)

        print("Training complete.")

    def _get_log_emission(self, tag_idx, word):

        if word in self.log_emission[tag_idx]:
            return self.log_emission[tag_idx][word]
        else:
            return self.log_emission_oov[tag_idx]

    def predict(self, sentence):

        if not sentence:
            return []

        T = len(sentence)          # Sentence length
        N = len(self.tags)         # Number of tags

        viterbi = np.full((T, N), -np.inf)
        backpointer = np.zeros((T, N), dtype=np.int32)

        word = sentence[0]
        is_oov = word not in self.vocab

        for j in range(N):
            if is_oov:
                # For OOV words: boost the tag suggested by suffix heuristic
                guessed_tag = self._guess_tag_from_suffix(word, position_in_sentence=0)
                if self.tags[j] == guessed_tag or (guessed_tag == 'PROPN' and self.tags[j] == 'NOUN'):
                    # Give higher emission probability to the heuristic tag
                    emission_log_prob = math.log(0.5)  # Strong prior for guessed tag
                else:
                    # Distribute remaining probability across other tags
                    emission_log_prob = math.log(0.5 / (N - 1))
            else:
                emission_log_prob = self._get_log_emission(j, word)

            viterbi[0, j] = self.log_initial[j] + emission_log_prob

        for t in range(1, T):
            word = sentence[t]
            is_oov = word not in self.vocab

            for j in range(N):
                if is_oov:
                    guessed_tag = self._guess_tag_from_suffix(word, position_in_sentence=t)
                    if self.tags[j] == guessed_tag or (guessed_tag == 'PROPN' and self.tags[j] == 'NOUN'):
                        emission_log_prob = math.log(0.5)
                    else:
                        emission_log_prob = math.log(0.5 / (N - 1))
                else:
                    emission_log_prob = self._get_log_emission(j, word)


                scores = viterbi[t - 1, :] + self.log_transition[:, j]
                best_prev = np.argmax(scores)

                viterbi[t, j] = scores[best_prev] + emission_log_prob
                backpointer[t, j] = best_prev

        best_last_tag = np.argmax(viterbi[T - 1, :])

        best_path = [0] * T
        best_path[T - 1] = best_last_tag
        for t in range(T - 2, -1, -1):
            best_path[t] = backpointer[t + 1, best_path[t + 1]]

        return [self.tags[idx] for idx in best_path]


In [ ]:
hmm = HMMTagger()
hmm.train(train_sents)

Training HMM on 3131 sentences...
  Vocabulary size: 10,983
  Number of tags:  12
  Tags: ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']
Training complete.


### Transition Matrix

In [ ]:
import pandas as pd

transition_matrix_df = pd.DataFrame(
    np.exp(hmm.log_transition), # Convert log probabilities back to probabilities
    index=hmm.tags,
    columns=hmm.tags
)

print("Transition Matrix:")
display(transition_matrix_df.round(4))

Transition Matrix:


,.,ADJ,ADP,ADV,CONJ,DET,NOUN,NUM,PRON,PRT,VERB,X
.,0.0985,0.0434,0.0749,0.0506,0.0598,0.1412,0.1916,0.1170,0.0609,0.0035,0.1294,0.0293
ADJ,0.0654,0.0670,0.0782,0.0049,0.0173,0.0052,0.6944,0.0215,0.0010,0.0116,0.0118,0.0217
ADP,0.0390,0.1085,0.0161,0.0142,0.0009,0.3242,0.3187,0.0643,0.0686,0.0018,0.0086,0.0352
ADV,0.1358,0.1330,0.1164,0.0797,0.0063,0.0675,0.0304,0.0304,0.0154,0.0138,0.3473,0.0241
CONJ,0.0342,0.1167,0.0548,0.0510,0.0005,0.1188,0.3516,0.0402,0.0597,0.0060,0.1568,0.0098
DET,0.0179,0.2023,0.0097,0.0126,0.0006,0.0058,0.6391,0.0227,0.0036,0.0004,0.0404,0.0449
NOUN,0.2419,0.0126,0.1771,0.0166,0.0438,0.0129,0.2631,0.0096,0.0044,0.0435,0.1455,0.0290
NUM,0.1141,0.0333,0.0358,0.0031,0.0134,0.0031,0.3551,0.1867,0.0021,0.0282,0.0168,0.2083
PRON,0.0438,0.0709,0.0199,0.0352,0.0050,0.0099,0.2046,0.0081,0.0095,0.0136,0.4851,0.0944
PRT,0.0444,0.0829,0.0204,0.0108,0.0019,0.1030,0.2415,0.0594,0.0185,0.0023,0.3993,0.0154


In [ ]:
for tag_name in ['VERB', 'NOUN', 'ADJ', 'ADV']:
    tag_idx = hmm.tag_to_idx[tag_name]
    # Sort words by emission count
    word_counts = sorted(hmm.emission_counts[tag_idx].items(), key=lambda x: -x[1])
    top5 = word_counts[:5]
    words_str = ", ".join(f"{w} ({int(c)})" for w, c in top5)
    print(f"Top-5 emissions for {tag_name:5s}: {words_str}")

Top-5 emissions for VERB : is (548), said (514), are (298), was (292), has (278)
Top-5 emissions for NOUN : % (367), Mr. (299), company (211), U.S. (179), year (173)
Top-5 emissions for ADJ  : new (127), other (118), more (89), last (75), many (70)
Top-5 emissions for ADV  : n't (261), also (112), not (112), more (66), even (56)


### Testing

In [ ]:
def evaluate(hmm_model, test_sentences):

    correct = 0
    total = 0

    tag_correct = Counter()
    tag_total = Counter()

    # Confusion matrix
    confusion = defaultdict(Counter)  # confusion[gold][predicted] = count

    # OOV tracking
    oov_correct = 0
    oov_total = 0

    for sent in test_sentences:
        words = [w for w, t in sent]
        gold_tags = [t for w, t in sent]
        pred_tags = hmm_model.predict(words)

        for i, (gold, pred) in enumerate(zip(gold_tags, pred_tags)):
            total += 1
            tag_total[gold] += 1
            confusion[gold][pred] += 1

            is_oov = words[i] not in hmm_model.vocab
            if is_oov:
                oov_total += 1

            if gold == pred:
                correct += 1
                tag_correct[gold] += 1
                if is_oov:
                    oov_correct += 1

    accuracy = correct / total if total > 0 else 0.0
    return accuracy, tag_correct, tag_total, confusion, oov_correct, oov_total

In [ ]:
accuracy, tag_correct, tag_total, confusion, oov_correct, oov_total = evaluate(hmm, test_sents)

print(f"  Overall Token-Level Accuracy: {accuracy:.4f}")
print(f"  OOV tokens: {oov_total} / {sum(tag_total.values())}")
print(f"  OOV accuracy: {oov_correct/oov_total:.4f}")

  Overall Token-Level Accuracy: 0.9020
  OOV tokens: 1515 / 19655
  OOV accuracy: 0.6013


In [ ]:
tag_accuracy_data = []
for tag in sorted(tag_total.keys()):
    tc = tag_correct[tag]
    tt = tag_total[tag]
    acc = tc / tt if tt > 0 else 0
    tag_accuracy_data.append({
        'Tag': tag,
        'Correct': tc,
        'Total': tt,
        'Accuracy': acc
    })

tag_accuracy_df = pd.DataFrame(tag_accuracy_data)

print("Per-Tag Accuracy DataFrame:")
display(tag_accuracy_df.round(4))

Per-Tag Accuracy DataFrame:


,Tag,Correct,Total,Accuracy
0,.,2255,2260,0.9978
1,ADJ,841,1256,0.6696
2,ADP,1868,1933,0.9664
3,ADV,444,648,0.6852
4,CONJ,428,434,0.9862
5,DET,1631,1699,0.9600
6,NOUN,5318,5665,0.9387
7,NUM,427,647,0.6600
8,PRON,511,535,0.9551
9,PRT,596,639,0.9327


In [ ]:
# Find top confusions
errors = []
for gold_tag, pred_counts in confusion.items():
    for pred_tag, count in pred_counts.items():
        if gold_tag != pred_tag:
            errors.append((gold_tag, pred_tag, count))

errors.sort(key=lambda x: -x[2])

print("Top 10 Confusion Pairs:")
print(f"  {'Gold':>8s} {'Predicted':>8s}  {'Count':>6s}")
for gold, pred, count in errors[:10]:
    print(f"  {gold:>8s} {pred:>8s}  {count:>6d}")

Top 10 Confusion Pairs:
      Gold Predicted   Count
       ADJ     NOUN     276
      VERB     NOUN     239
       NUM     NOUN     145
      NOUN      DET     111
      NOUN     VERB      92
         X     NOUN      78
       ADV      ADP      57
      VERB      ADP      50
       DET      ADP      50
      VERB      DET      47


In [ ]:
demo_sentences = [
    "The cat sat on the mat".split(),
    "Running quickly is surprisingly difficult".split(),
    "The keys to the cabinet are on the table".split(),    # Long-range agreement
    "I need to book a flight to New York".split(),          # Ambiguous word: 'book'
    "She decided to run the marathon yesterday".split(),    # Ambiguous word: 'run'
]

for sent_words in demo_sentences:
    pred = hmm.predict(sent_words)
    tagged = " ".join(f"{w}/{t}" for w, t in zip(sent_words, pred))
    print(f"  {tagged}")
    print()

  The/DET cat/NOUN sat/NOUN on/ADP the/DET mat/NOUN

  Running/VERB quickly/ADV is/VERB surprisingly/ADV difficult/ADJ

  The/DET keys/NOUN to/PRT the/DET cabinet/NOUN are/VERB on/ADP the/DET table/NOUN

  I/PRON need/VERB to/PRT book/VERB a/DET flight/NOUN to/PRT New/NOUN York/NOUN

  She/PRON decided/VERB to/PRT run/VERB the/DET marathon/NOUN yesterday/NOUN

